In [8]:
"""
Example script to run the electricity model.
It consists of three parts:
1. Generation
2. Grid
3. Storage
"""

import pandas as pd
import numpy as np
import xarray as xr
import scipy
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA
from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid
)

from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks, Maintenance, MaterialIntensities, SharesInflowStocks
from imagematerials.factory import ModelFactory, Sector
from imagematerials.preprocessing import get_preprocessing_data
import prism

VARIANT = "VLHO"
SCEN = "SSP2"
scen_folder = SCEN + "_" + VARIANT
# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used (?) 
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
#new
path_base = Path(path_base, "data", "raw")

# create the folder out_test if it does not exist
if not (path_base / 'imagematerials' / 'electricity' / 'out_test').is_dir():
    (path_base / 'imagematerials' / 'electricity' / 'out_test').mkdir(parents=True)

YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting

In [2]:
path_base


WindowsPath('C:/Projects/image-materials/data/raw')

In [9]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                #  "SSP2_VLHO":("SSP2_VLHO", None)
                 }

scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    
    factory = ModelFactory(
    elc_sector, complete_timeline
    ).add(GenericStocks, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add"
                          ]
    ).add(MaterialIntensities, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add"
                          ]
    )
    model = factory.finish()

    model.simulate(simulation_timeline)

grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table


# Generation -------------------------------------------------------

In [10]:
climate_policy_scenario_dir = Path(path_base, "image", scen_folder) #, "EnergyServices")
sec_electr_gen = get_preprocessing_data("electricity",base_dir=path_base,climate_policy_scenario_dir=climate_policy_scenario_dir, standard_scenario=scen_folder, year_start=YEAR_START, year_end=YEAR_END, year_out=YEAR_OUT)

AssertionError: 

In [ ]:
# Define the complete timeline, including historic tail
time_start = sec_electr_gen.prep_data["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

#sec_electr_gen = Sector("electr_gen", prep_data)

factory = ModelFactory(
    sec_electr_gen, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model = factory.finish()

model.simulate(simulation_timeline)

list(model.electricity)

In [ ]:
list(model.electricity)

# Grid -------------------------------------------------------------

In [ ]:
prep_data_lines, prep_data_add = get_preprocessing_data_grid(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)

## Lines ----

In [ ]:
# Define the complete timeline, including historic tail
time_start = prep_data_lines["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_grid_lines = Sector("electr_grid_lines", prep_data_lines)

factory = ModelFactory(
    sec_electr_grid_lines, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_lines = factory.finish()

model_lines.simulate(simulation_timeline)
list(model_lines.electr_grid_lines)

In [ ]:
model_lines.electr_grid_lines["stock_by_cohort_materials"]

## Grid Additions ----

In [ ]:
time_start = prep_data_add["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_grid_add = Sector("electr_grid_add", prep_data_add)

factory = ModelFactory(
    sec_electr_grid_add, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_add = factory.finish()

model_add.simulate(simulation_timeline)
list(model_add.electr_grid_add)

# Storage -------------------------------------------------------

In [ ]:
YEAR_FIRST_STOR = 1907 # first use of pumped storage was in 1907 at the Engeweiher pumped storage facility near Schaffhausen, Switzerland (Mitali et al. 2022)
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting

prep_data_phs, prep_data_oth_storage = get_preprocessing_data_stor(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)

In [ ]:
# Pumped Hydropower Storage (PHS) =====

# # Define the complete timeline, including historic tail
time_start = prep_data_phs["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_stor_phs = Sector("electr_stor_phs", prep_data_phs)

factory = ModelFactory(
    sec_electr_stor_phs, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_phs = factory.finish()

model_phs.simulate(simulation_timeline)
list(model_phs.electr_stor_phs)

In [ ]:
# Other Storage =====

time_start = prep_data_oth_storage["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1) #1970

sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

factory = ModelFactory(
    sec_electr_stor_oth, complete_timeline
    ).add(SharesInflowStocks
    ).add(MaterialIntensities
    )
model_oth = factory.finish()

model_oth.simulate(simulation_timeline)
list(model_oth.electr_stor_oth)